# 3. Science Frame Reduction and Light Curve Extraction

## Why process science frames?
Raw science images are affected by instrumental signatures that must be removed before we can measure precise stellar fluxes. The main corrections are:

- **Bias subtraction** – removes the electronic offset that is added to every pixel, ensuring that the zero‑point of the CCD is consistent.
- **Flat‑field division** – corrects for pixel‑to‑pixel sensitivity variations, vignetting, and shadows from dust. After this step, a uniform source appears uniform across the frame.
- **Error propagation** – because every operation adds noise, we need to compute the uncertainty in each pixel after correction. This is essential for later photometric error estimates.

Once the images are cleaned, we convert the exposure times to a uniform time system. The **Barycentric Julian Date in the Barycentric Dynamical Time (BJD_TDB)** is the standard for exoplanet transit studies. It corrects for the finite speed of light and the Earth’s motion around the barycenter, so that transit times from different observatories can be compared directly.

## Objectives
1. **Apply bias and flat corrections** to all science frames, using the master bias and master flat created in the previous notebooks.
2. **Propagate uncertainties** through the corrections, so we have error maps for each image.
3. **Compute precise timestamps** – convert the mid‑exposure JD_UTC to BJD_TDB using the known observatory coordinates and target position.
4. **Perform aperture photometry** – measure the flux of the target star and several reference stars in each corrected image.
5. **Construct a differential light curve** – by dividing the target flux by a combination of reference fluxes, we cancel out transparency variations and other common‑mode effects.
6. **Visualise the light curve** – plot the differential magnitude versus BJD_TDB to reveal the transit of WASP‑12b.

A successful reduction yields a clean light curve that can later be modelled to derive the planetary parameters.

In [1]:
# -----Imports-----
import numpy as np
import matplotlib.pyplot as plt
import os
import certifi
os.environ['SSL_CERT_FILE'] = certifi.where()
from astropy.io import fits # Astropy FITS handling
import pickle

from astropy.time import Time
from astropy import coordinates as coord, units as u

# Plot style
import matplotlib as mpl
mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["EB Garamond", "Georgia", "Times New Roman"],
    # ---- Sizes ----
    "font.size": 12,
    "axes.titlesize": 13,
    "axes.labelsize": 13,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "axes.labelweight": "light",  # x/y labels
    "font.weight": "light",
    # ---- Axes/ticks style like the PDF ----
    "axes.edgecolor": "0.2",
    "axes.linewidth": 0.7,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.top": True,
    "ytick.right": True,
    "xtick.minor.visible": True,
    "ytick.minor.visible": True,
    "xtick.major.size": 4,
    "ytick.major.size": 4,
    "xtick.minor.size": 2,
    "ytick.minor.size": 2,
    "xtick.major.width": 0.5,
    "ytick.major.width": 0.5,
    "xtick.minor.width": 0.4,
    "ytick.minor.width": 0.4,

    "axes.grid": False,
    "axes.labelcolor": "0.2",
    "xtick.color": "0.2",
    "ytick.color": "0.2",
    "text.color": "0.2",
})


%matplotlib widget


## Process all science frames



In [2]:
# Use the full science list
science_list = np.genfromtxt('./group15_WASP-12_20191229/science/science.list',dtype=str)

science_test_list = science_list[:10] 



In [3]:
bias00_fits = fits.open('./group15_WASP-12_20191229/science/'+science_list[0])
bias00_hdu = bias00_fits[0]
bias00_hdu.header


SIMPLE  =                    T / file does conform to FITS standard             
BITPIX  =                   16 / number of bits per data pixel                  
NAXIS   =                    2 / number of data axes                            
NAXIS1  =                  521 / length of data axis 1                          
NAXIS2  =                  222 / length of data axis 2                          
EXTEND  =                    T / FITS dataset may contain extensions            
COMMENT   FITS (Flexible Image Transport System) format is defined in 'Astronomy
COMMENT   and Astrophysics', volume 376, page 359; bibcode: 2001A&A...376..359H 
BZERO   =                32768 / offset data range to that of unsigned short    
BSCALE  =                    1 / default scaling factor                         
DATE    = '2019-12-29T18:33:07' / file creation date (YYYY-MM-DDThh:mm:ss UT)   
FILENAME= 'AF578220.fits'      / Original file name                             
TIMESYS = 'UTC     '        

In [4]:
# Load needed files and set known parameters

median_bias = pickle.load(open("median_bias.p", "rb"))
median_normalized_flat =  pickle.load(open("median_normalized_flat.p", "rb"))
median_normalized_flat_errors =  pickle.load(open("median_normalized_flat_errors.p", "rb"))

bias_std = 1.31 # [e] photoelectrons
readout_noise = 7.10 # [e] photoelectrons
gain = 1.91 # [e/ADU]



In [5]:
for science_name in science_list:
    # Read raw science image
    with fits.open('./group15_WASP-12_20191229/science/' + science_name) as hdul:
        science_data = hdul[0].data * gain   # Convert to electrons

    # Bias subtraction
    science_debiased = science_data - median_bias

    # ---- Mask for valid flat pixels ----
    flat_mask = (median_normalized_flat > 0)

    # Flat correction: only where flat > 0
    science_corrected = np.zeros_like(science_debiased)
    science_corrected[flat_mask] = science_debiased[flat_mask] / median_normalized_flat[flat_mask]

    # ---- Error propagation ----
    # Debiased error (photon noise + readout noise + bias uncertainty)
    science_debiased_errors = np.sqrt(readout_noise**2 + bias_std**2 + np.abs(science_debiased))

    # Combined error: only where flat > 0 AND science_debiased != 0 (to avoid division by zero in the error formula)
    valid = flat_mask & (science_debiased != 0)
    science_corrected_errors = np.zeros_like(science_corrected)
    science_corrected_errors[valid] = science_corrected[valid] * np.sqrt(
        (science_debiased_errors[valid] / science_debiased[valid])**2 +
        (median_normalized_flat_errors[valid] / median_normalized_flat[valid])**2
    )
    # For pixels that are not valid, we leave error as 0 (or np.nan) – they will be ignored in photometry

    # Save results
    base = science_name[:-5]   # remove '.fits'
    pickle.dump(science_corrected, open(f'./group15_WASP-12_20191229/correct/{base}_corr.p', 'wb'))
    pickle.dump(science_corrected_errors, open(f'./group15_WASP-12_20191229/correct/{base}_corr_errors.p', 'wb'))

    
    
print("All science frames corrected and saved.")



All science frames corrected and saved.
